# Public reproduction copy

The analysis cells below are copied from the manuscript-generating notebook. Only filesystem configuration and output locations have been changed. The Benjamini-Hochberg correction is evaluated over the same 11-dataset family as in the manuscript analysis; the final forest plot displays the four prespecified factors.


# A) 最小修正：先行・遅れ（時系列の向き）を頑健に評価する cross-correlation

このノートブックは、学生版 `Cell 7: Statistical Analysis (Correlation)` を **「先行・遅れの向き」目的に最適化**して、以下を追加します。

- **補間をやめる**（欠損は NaN のまま計算）
- **ベースライン補正を bootstrap の内側で再推定**（Coulon & Larson Fig.4 の流れに合わせる）
- **非対称性指標 A**（正ラグ優位か負ラグ優位か）と **CI（信頼区間）**を算出
- 図は **Arial** フォントで作図し、**PDF** で保存

> 注意：ここでは **ローカルデトレンド + z-score 後**に相関を取ります。したがって **振幅比較（amplitude comparison）はしません**。

In [ ]:
# Cell A0: Imports & plotting defaults (Arial + PDF)
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'analyses').is_dir() and (candidate / 'README.md').is_file():
            return candidate
    return start

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
GAO2026_DATA_ROOT = Path(os.environ.get('GAO2026_DATA_ROOT', REPO_ROOT / 'data')).expanduser().resolve()
GAO2026_OUTPUT_ROOT = Path(os.environ.get('GAO2026_OUTPUT_ROOT', REPO_ROOT / 'outputs')).expanduser().resolve()
CSV_ROOT = GAO2026_DATA_ROOT / '02_Cross_correlation' / 'csv'
ANALYSIS_OUTPUT_ROOT = GAO2026_OUTPUT_ROOT / '02_Cross_correlation'

# --- plotting: Arial + vector-friendly PDF embedding ---
plt.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,   # TrueType
    "ps.fonttype": 42,
    "figure.dpi": 150,
})

def ensure_outdir(base="output_A"):
    outdir = ANALYSIS_OUTPUT_ROOT / base
    outdir.mkdir(parents=True, exist_ok=True)
    return outdir

print("Ready. Matplotlib font:", plt.rcParams["font.family"])

## A1. 入力CSVとパラメータ

- `CSV_PATHS` に **all_cells_spot_intensity_seg_cell.csv** などを指定してください。
- `frame` / `cell_id` / `ch1_mean` / `ch2_mean`（必要なら `ch3_mean`）がある前提です（学生ノートと同じ）。
- `dt_min` は「1フレームが何分か」を入れておくと、図で分表示できます（任意）。

In [ ]:
# Cell A1: User settings (edit here)


CSV_PATHS = [
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_BRD4.csv", "BRD4"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_CHD4.csv", "CHD4"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_H3K27ac.csv", "H3K27ac"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_HDAC1.csv", "HDAC1"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_HDAC3.csv", "HDAC3"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_HDAC3_del_detrend.csv", "HDAC3_de"),
    (CSV_ROOT / "all_cells_spot_intensity_P300.csv", "P300"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SIN3A.csv", "SIN3A"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SOX2.csv", "SOX2"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SOX2_del_detrend.csv", "SOX2_de"),
    (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SOX2_1min.csv", "SOX2_1min"),
]

CONFIG = {
    # --- QC / lags ---
    "cell_min_points": 8,          # minimum frames per cell to include
    "pair_thresh": 3,              # minimum overlapping pairs to compute a lag point
    "max_lag": 10,                 # +/- lag in frames

    # --- detrending / standardization ---
    "apply_detrend": True,
    "detrend_method": "median",    # "median" or "mean"
    "detrend_window": 15,          # in frames
    "use_zscore": True,            # -> correlation-coefficient-like (amplitude is not preserved)

    # --- smoothing (optional) ---
    "use_multitau_smoothing": True,
    "multitau_scheme": [(0, 3, 1), (4, 7, 3), (8, 10, 5)],

    # --- baseline correction ---
    # Choose a range where the curve is expected to be at a plateau (|tau| in [base_min, base_max]).
    # IMPORTANT: This must fit within max_lag.
    "base_lag_range": (7, 10),

    # --- bootstrap ---
    "boot_n": 3000,
    "ci_percentiles": (2.5, 97.5),

    # --- display options ---
    "dt_min": 2.0,                 # minutes per frame (set None to skip minute axis)
}

LAG_MASTER = np.arange(-CONFIG["max_lag"], CONFIG["max_lag"] + 1, dtype=int)
OUTDIR = ensure_outdir()

print("LAG_MASTER:", LAG_MASTER)
print("Output dir:", OUTDIR.resolve())

## A2. ヘルパー関数（補間なし・NaN対応）

- 欠損フレームは **NaN のまま保持**します（補間しません）。
- 相関は **overhang trimming** + **有限値ペアのみ**で計算します。
- ローカルデトレンドは rolling median/mean による **遅いトレンド除去**（学生版と同系統）。

In [ ]:
# Cell A2: Helper functions (no interpolation; NaNs are kept)

def zscore(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a, float)
    m = np.nanmean(a)
    s = np.nanstd(a)
    if not np.isfinite(s) or s <= 0:
        return np.full_like(a, np.nan)  # keep NaN structure
    return (a - m) / s

def local_detrend(a: np.ndarray, method="median", window=15) -> np.ndarray:
    a = np.asarray(a, float)
    s = pd.Series(a)
    if method == "median":
        base = s.rolling(window=window, center=True, min_periods=1).median()
    else:
        base = s.rolling(window=window, center=True, min_periods=1).mean()
    base = base.to_numpy(dtype=float)
    return a - base

def cross_correlate_direct(x: np.ndarray, y: np.ndarray, pair_thresh: int = 1):
    """
    Returns:
      lags: integer lags from -(n-1) .. +(n-1)
      rho:  mean of x(t)*y(t+lag) on overlapping finite pairs
      n_pair: number of finite pairs used at each lag
    Convention:
      lag < 0 means x leads (y is shifted forward).
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    n = len(x)
    lags = np.arange(-n + 1, n, dtype=int)
    rho = np.full(lags.shape, np.nan, dtype=float)
    n_pair = np.zeros_like(rho, dtype=float)

    for i, lag in enumerate(lags):
        if lag < 0:
            xi, yi = x[:lag], y[-lag:]
        elif lag > 0:
            xi, yi = x[lag:], y[:-lag]
        else:
            xi, yi = x, y
        mask = np.isfinite(xi) & np.isfinite(yi)
        k = int(mask.sum())
        n_pair[i] = k
        if k >= pair_thresh:
            rho[i] = float(np.mean(xi[mask] * yi[mask]))
    return lags, rho, n_pair

def map_to_master(lags, values):
    lut = dict(zip(lags, values))
    out = np.full(LAG_MASTER.shape, np.nan, dtype=float)
    for j, lg in enumerate(LAG_MASTER):
        if lg in lut:
            out[j] = lut[lg]
    return out

def variable_bin_smooth(G, W, scheme):
    """Weighted moving-average with variable window size by |tau|."""
    G = np.asarray(G, float)
    W = np.asarray(W, float)
    out = np.full_like(G, np.nan, dtype=float)

    def win_for_tau(tau_abs):
        for a, b, w in scheme:
            if a <= tau_abs <= b:
                return max(1, int(w))
        return 1

    n = len(G)
    for i, tau in enumerate(LAG_MASTER):
        w = win_for_tau(abs(int(tau)))
        if w <= 1:
            out[i] = G[i]
            continue
        half = w // 2
        lo, hi = max(0, i - half), min(n, i + half + 1)
        ww, gg = W[lo:hi], G[lo:hi]
        if np.isfinite(ww).any() and np.nansum(ww) > 0:
            out[i] = np.nansum(gg * ww) / np.nansum(ww)
    return out

def baseline_offset(G, W_sum, base_lag_range):
    """Weighted mean of G at large |tau| region, used as baseline offset."""
    base_min, base_max = base_lag_range
    base_min = int(base_min); base_max = int(base_max)
    # Clamp to available lag range
    base_max = min(base_max, int(np.max(np.abs(LAG_MASTER))))
    base_min = min(base_min, base_max)
    mask = (
        (np.abs(LAG_MASTER) >= base_min)
        & (np.abs(LAG_MASTER) <= base_max)
        & np.isfinite(G)
        & (W_sum > 0)
    )
    if not np.any(mask):
        return 0.0
    return float(np.nansum(G[mask] * W_sum[mask]) / np.nansum(W_sum[mask]))

def asymmetry_index(G, W_sum=None):
    """
    A = mean_{tau>0}(G) - mean_{tau<0}(G)
    If W_sum is provided, use weighted means.
    """
    G = np.asarray(G, float)
    tau = LAG_MASTER
    pos = tau > 0
    neg = tau < 0
    if W_sum is None:
        mp = np.nanmean(G[pos])
        mn = np.nanmean(G[neg])
    else:
        W = np.asarray(W_sum, float)
        mp = np.nansum(G[pos] * W[pos]) / np.nansum(W[pos]) if np.nansum(W[pos]) > 0 else np.nan
        mn = np.nansum(G[neg] * W[neg]) / np.nansum(W[neg]) if np.nansum(W[neg]) > 0 else np.nan
    return float(mp - mn)

print("Helpers defined.")

## A3. 相関行列の作成（細胞ごと・補間なし）

学生版では `reindex(...).interpolate()` が入っていましたが、ここでは **補間しません**。

- `frames` の範囲は各 cell の最小〜最大フレームに合わせます
- 欠損フレームは NaN のまま
- detrend と z-score は（必要なら）ここで適用

In [ ]:
# Cell A3: Build per-cell correlation matrices (NO interpolation)

def pick_trace_no_interp(cell_df: pd.DataFrame, frames: np.ndarray, prefer: str, fallback: str):
    """Return values aligned to 'frames' without interpolation; missing frames => NaN."""
    if prefer in cell_df.columns:
        col = prefer
    elif fallback in cell_df.columns:
        col = fallback
    else:
        return None, None
    vals = cell_df.set_index("frame")[col].reindex(frames).to_numpy(dtype=float)
    return vals, col

def calc_cc_matrices_no_interp(df: pd.DataFrame, col_x_prefix: str, col_y_prefix: str):
    cc_list, wt_list = [], []
    acx_list, wtx_list = [], []
    acy_list, wty_list = [], []
    cell_ids = []

    x_mean = f"{col_x_prefix}_mean"
    y_mean = f"{col_y_prefix}_mean"
    x_det = f"{col_x_prefix}_detrended"
    y_det = f"{col_y_prefix}_detrended"

    for cid, sub in df.groupby("cell_id"):
        sub = sub.sort_values("frame")
        frames = np.arange(int(sub.frame.min()), int(sub.frame.max()) + 1, dtype=int)

        if len(frames) < CONFIG["cell_min_points"]:
            continue

        x_raw, x_col = pick_trace_no_interp(sub, frames, x_det, x_mean)
        y_raw, y_col = pick_trace_no_interp(sub, frames, y_det, y_mean)
        if x_raw is None or y_raw is None:
            continue

        # optional detrend if we only have *_mean columns
        if CONFIG["apply_detrend"]:
            if x_col == x_mean:
                x_raw = local_detrend(x_raw, CONFIG["detrend_method"], CONFIG["detrend_window"])
            if y_col == y_mean:
                y_raw = local_detrend(y_raw, CONFIG["detrend_method"], CONFIG["detrend_window"])

        # standardize (z-score) if requested
        x = zscore(x_raw) if CONFIG["use_zscore"] else x_raw
        y = zscore(y_raw) if CONFIG["use_zscore"] else y_raw

        # per-cell correlations
        lags, cc, w = cross_correlate_direct(x, y, pair_thresh=CONFIG["pair_thresh"])
        cc_list.append(map_to_master(lags, cc))
        wt_list.append(map_to_master(lags, w))

        lags, acx, w = cross_correlate_direct(x, x, pair_thresh=CONFIG["pair_thresh"])
        acx_list.append(map_to_master(lags, acx))
        wtx_list.append(map_to_master(lags, w))

        lags, acy, w = cross_correlate_direct(y, y, pair_thresh=CONFIG["pair_thresh"])
        acy_list.append(map_to_master(lags, acy))
        wty_list.append(map_to_master(lags, w))

        cell_ids.append(cid)

    if len(cell_ids) == 0:
        return None

    return {
        "cell_ids": cell_ids,
        "cc_xy": np.vstack(cc_list),
        "wt_xy": np.vstack(wt_list),
        "ac_x": np.vstack(acx_list),
        "wt_x": np.vstack(wtx_list),
        "ac_y": np.vstack(acy_list),
        "wt_y": np.vstack(wty_list),
    }

print("Matrix builder ready.")

## A4. グループ平均 + ブートストラップ（ベースライン補正を内側で再推定）

Coulon & Larson の推奨（Fig.4）に合わせて、**bootstrap の各反復で**：

1. resample（cell単位で復元抽出）
2. 重み付き平均で G(τ) を作る
3. **ベースライン offset を推定して差し引く**
4.（必要なら）multi-tau 平滑化
5. 分布から CI を計算

さらに **非対称性指標 A** を bootstrap 分布から CI を出します。

In [ ]:
# Cell A4: Group mean with bootstrap baseline correction + A (asymmetry) with CI

def weighted_mean_curve(mat, wt):
    # mat/wt shape: (n_cells, n_lags)
    W = np.nansum(wt, axis=0)
    num = np.nansum(mat * wt, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        G = num / np.maximum(W, 1e-12)
    return G, W

def compute_curve_with_corrections(mat, wt):
    """Compute group curve G and weight W, then baseline-correct and (optionally) smooth."""
    G, W = weighted_mean_curve(mat, wt)
    off = baseline_offset(G, W, CONFIG["base_lag_range"])
    G = G - off
    if CONFIG["use_multitau_smoothing"]:
        G = variable_bin_smooth(G, W, CONFIG["multitau_scheme"])
    return G, W, off

def bootstrap_group(mat, wt, seed=123):
    rng = np.random.default_rng(seed)
    n_cells = mat.shape[0]
    B = int(CONFIG["boot_n"])
    boot = np.empty((B, mat.shape[1]), dtype=float)
    Avals = np.empty(B, dtype=float)

    for i in range(B):
        sel = rng.choice(n_cells, size=n_cells, replace=True)
        mat_b = mat[sel]
        wt_b = wt[sel]
        Gb, Wb, _ = compute_curve_with_corrections(mat_b, wt_b)
        boot[i] = Gb
        Avals[i] = asymmetry_index(Gb, W_sum=Wb)

    lo, hi = CONFIG["ci_percentiles"]
    ci_lo = np.nanpercentile(boot, lo, axis=0)
    ci_hi = np.nanpercentile(boot, hi, axis=0)
    G_mean = np.nanmean(boot, axis=0)  # bootstrap mean
    sem = np.nanstd(boot, axis=0)      # standard error per Fig.4 logic

    A_ci = (float(np.nanpercentile(Avals, lo)), float(np.nanpercentile(Avals, hi)))
    A_mean = float(np.nanmean(Avals))

    return {
        "G_mean": G_mean,
        "ci_lo": ci_lo,
        "ci_hi": ci_hi,
        "sem": sem,
        "A_mean": A_mean,
        "A_ci": A_ci,
        "boot": boot,
        "Avals": Avals,
    }

print("Bootstrap functions ready.")

## A5. 図（PDF出力）と実行

- 左：SNAP と MCP の autocorrelation（両方）
- 右：SNAP vs MCP の cross-correlation（**負のτが「SNAPが先行」**）
- CI は bootstrap 分布の percentiles（デフォルト 95% CI）
- 追加で、非対称性指標 A と CI を `metrics_A.csv` に保存

In [ ]:
# Cell A5: Plotting + runner (PDF output)

def make_time_axis(ax, dt_min):
    """Optionally add a secondary x-axis in minutes."""
    if dt_min is None:
        return
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ticks = ax.get_xticks()
    ax2.set_xticks(ticks)
    ax2.set_xticklabels([f"{t*dt_min:.0f}" for t in ticks])
    ax2.set_xlabel("Lag τ (min)")

def plot_ac_cc(label, gx, gx_ci, gy, gy_ci, gxy, gxy_ci, n_cells, outpath_pdf):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # --- autocorr panel ---
    ax = axes[0]
    ax.fill_between(LAG_MASTER, gx_ci[0], gx_ci[1], alpha=0.2)
    ax.plot(LAG_MASTER, gx, label="SNAP Auto")
    ax.fill_between(LAG_MASTER, gy_ci[0], gy_ci[1], alpha=0.2)
    ax.plot(LAG_MASTER, gy, label="MCP Auto")
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_xlabel("Lag τ (frames)")
    ax.set_ylabel("Correlation (z-scored)")
    ax.set_title(f"Autocorrelations (N={n_cells})")
    ax.legend()
    ax.grid(axis="y", linestyle=":", alpha=0.6)
    make_time_axis(ax, CONFIG.get("dt_min", None))

    # --- cross-corr panel ---
    ax = axes[1]
    ax.fill_between(LAG_MASTER, gxy_ci[0], gxy_ci[1], alpha=0.2)
    ax.plot(LAG_MASTER, gxy, label="SNAP vs MCP")
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.axvline(0, color="black", lw=0.8, ls=":")
    ax.set_xlabel("Lag τ (frames)")
    ax.set_ylabel("Cross-correlation (z-scored)")
    ax.set_title("Cross-correlation\nNeg τ: SNAP leads")
    ax.legend()
    ax.grid(axis="y", linestyle=":", alpha=0.6)
    make_time_axis(ax, CONFIG.get("dt_min", None))

    fig.suptitle(label)
    fig.tight_layout()
    fig.savefig(outpath_pdf, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def run_A():
    metrics_rows = []
    for csv_path, label in CSV_PATHS:
        print(f"\n[Processing A] {label}")
        df = pd.read_csv(csv_path)

        res = calc_cc_matrices_no_interp(df, "ch1", "ch2")  # ch1=SNAP, ch2=MCP
        if res is None:
            print("  -> No valid cells.")
            continue

        n_cells = len(res["cell_ids"])
        print("  N cells:", n_cells)

        # autocorr SNAP, autocorr MCP, crosscorr SNAP vs MCP
        boot_x = bootstrap_group(res["ac_x"], res["wt_x"], seed=123)
        boot_y = bootstrap_group(res["ac_y"], res["wt_y"], seed=456)
        boot_xy = bootstrap_group(res["cc_xy"], res["wt_xy"], seed=789)

        out_pdf = OUTDIR / f"A_{label}_AC_CC.pdf"
        plot_ac_cc(
            label=label,
            gx=boot_x["G_mean"], gx_ci=(boot_x["ci_lo"], boot_x["ci_hi"]),
            gy=boot_y["G_mean"], gy_ci=(boot_y["ci_lo"], boot_y["ci_hi"]),
            gxy=boot_xy["G_mean"], gxy_ci=(boot_xy["ci_lo"], boot_xy["ci_hi"]),
            n_cells=n_cells,
            outpath_pdf=out_pdf,
        )
        print("  Saved:", out_pdf)

        metrics_rows.append({
            "label": label,
            "csv_path": csv_path,
            "N_cells": n_cells,
            "A_mean": boot_xy["A_mean"],
            "A_ci_low": boot_xy["A_ci"][0],
            "A_ci_high": boot_xy["A_ci"][1],
            "max_lag_frames": CONFIG["max_lag"],
            "base_lag_range": str(CONFIG["base_lag_range"]),
            "boot_n": CONFIG["boot_n"],
            "use_zscore": CONFIG["use_zscore"],
            "detrend": CONFIG["apply_detrend"],
        })

    if metrics_rows:
        out_csv = OUTDIR / "metrics_A.csv"
        pd.DataFrame(metrics_rows).to_csv(out_csv, index=False)
        print("\n[DONE A] wrote:", out_csv.resolve())
    else:
        print("\n[DONE A] no outputs (no datasets?)")

run_A()

# Timelapse_mod 追加解析：LLI と Event-aligned 解析

このノートは **Timelapse_mod.pdf の前処理方針（補間なし / NaN 対応 / rolling detrend + within-cell z-score）** に合わせて、以下を追加で実行します。

1. **Lead/Lag Index (LLI)**  
   - per-cell のクロス相関曲線（ベースライン補正後）から LLI を算出  
   - **時間シフト置換検定**で「負ラグ優位 / 正ラグ優位」が統計的に偏っているかを評価

2. **Event-aligned 解析（state の OFF→ON / ON→OFF）**  
   - `state` の遷移をイベントとして、SNAP（例：`ch1`）をアライン  
   - 前後の平均差（pre/post metric）を per-cell で作り、Wilcoxon + bootstrap CI で評価

---

## 前提（CSV）
- `cell_id`, `frame`
- signal: `ch1_mean`/`ch1_detrended`, `ch2_mean`/`ch2_detrended`（必要に応じて `ch3_*`）
- event-aligned 解析をする場合：`state`（例：`ON` / `OFF`）

---

## 出力
- `output_LLI/` : LLI の図（PDF）と summary CSV
- `output_EVT/` : event-aligned 図（PDF）と summary CSV

> 既に Timelapse_mod のノートブックを実行済みの場合は、`CSV_PATHS` と `CONFIG` をそのまま使えます。  
> このノートは単独でも動くように最低限のデフォルトも用意しています。


In [ ]:
# --- Imports ---
import os
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# --- plotting: Arial + vector-friendly PDF embedding ---
mpl.rcParams.update({
    "font.family": "Arial",
    "pdf.fonttype": 42,  # TrueType
    "ps.fonttype": 42,
    "figure.dpi": 150,
})

# optional stats
try:
    from scipy.stats import wilcoxon
except Exception:
    wilcoxon = None

print("Ready. Matplotlib font:", mpl.rcParams["font.family"])


## Q) detrend はカラム構成によらず常に同じ方式？

**「detrend の“アルゴリズム”自体は同じ」です。**  
Timelapse_mod では rolling median/mean（`local_detrend`）でローカルトレンドを引く設計です。

ただし、**どの列を入力に使うか**で実際の処理が変わり得ます：

- `ch1_detrended` が CSV に存在する場合：  
  Timelapse_mod（A の実装）では **detrended 列を優先して使い**、その場合は追加の detrend を行わない（＝既に detrend 済みとして扱う）設計でした。
- `ch1_detrended` が無い場合：  
  `ch1_mean` を使って **`local_detrend` を適用**します。

つまり、データセット間で `*_detrended` の作り方が完全に同一でない可能性があるなら、  
**「結果としての detrend が “カラム構成に依存せず完全に同一” とは保証されません」。**

---

### どうしたいかで選択肢
このノートの `SOURCE_MODE` で切り替えできます。

- `SOURCE_MODE="prefer_detrended"`（Timelapse_mod と同じ挙動）  
  - `*_detrended` があればそれを優先  
  - 無ければ `*_mean` + `local_detrend`
- `SOURCE_MODE="prefer_mean"`（“統一した detrend” をしたい場合）  
  - `*_mean` があれば常にそれを使用し、必ず `local_detrend` を適用  
  - `*_mean` が無いときだけ `*_detrended` を使用（この場合の追加 detrend は既定ではしません）

> どちらが正しい/誤りというより、**解析の再現性（統一性）を優先するか**、  
> **既存の `*_detrended` を信頼して踏襲するか**の選択です。


In [ ]:
# ============================================================
# 0) Settings (edit here)
# ============================================================

# If Timelapse_mod already defined these, reuse them.
if "CSV_PATHS" not in globals():
    CSV_PATHS = [
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_BRD4.csv", "BRD4"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_CHD4.csv", "CHD4"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_H3K27ac.csv", "H3K27ac"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_HDAC1.csv", "HDAC1"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_HDAC3.csv", "HDAC3"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_HDAC3_del_detrend.csv", "HDAC3_de"),
        (CSV_ROOT / "all_cells_spot_intensity_P300.csv", "P300"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SIN3A.csv", "SIN3A"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SOX2.csv", "SOX2"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SOX2_del_detrend.csv", "SOX2_de"),
        (CSV_ROOT / "all_cells_spot_intensity_seg_cell_SOX2_1min.csv", "SOX2_1min"),
    ]

if "CONFIG" not in globals():
    # Minimal default (match Timelapse_mod A1 style as much as possible)
    CONFIG = {
        "cell_min_points": 8,
        "pair_thresh": 3,
        "max_lag": 10,
        "apply_detrend": True,
        "detrend_method": "median",  # 'median' or 'mean'
        "detrend_window": 15,
        "detrend_mode": "sub",       # 'sub' recommended
        "use_zscore": True,
        "base_lag_range": (7, 10),   # absolute lag range used for baseline offset
        "boot_n": 3000,
        "dt_min": 2.0,               # for plotting only; set None if unknown
    }

# --- Add-on analysis parameters ---
ADDON = {
    # column selection policy
    "SOURCE_MODE": "prefer_detrended",  # "prefer_detrended" (Timelapse_mod) or "prefer_mean" (unified)
    "FORCE_DETREND": False,             # if True, detrend even when using *_detrended (usually False)

    # LLI parameters
    "LLI_WINDOW_FRAMES": 5,             # use lags [-5..-1] vs [1..5]
    "B_PERM": 1000,                     # time-shift permutations
    "B_BOOT": int(CONFIG.get("boot_n", 3000)),  # bootstrap for CI of median/statistics
    "SEED": 12345,

    # event-aligned parameters
    "EVENT_WINDOW_FRAMES": int(CONFIG.get("max_lag", 10)),  # +/- window around event
    "MIN_ON_LEN": 5,                 # run-length filter (frames)
    "MIN_OFF_LEN": 5,                # run-length filter (frames)
    "PRE_WIN_FRAMES": (-3, -1),      # pre window (frames, negative)
    "POST_WIN_FRAMES": (1, 3),       # post window (frames, positive)
    "EVENT_MIN_FRACTION_FINITE": 0.7, # require >=70% finite points in an aligned segment

    # state labels (event-aligned)
    "STATE_COL": "state",
    "ON_LABEL": "ON",
    }

OUTDIR_LLI = ANALYSIS_OUTPUT_ROOT / "output_LLI"
OUTDIR_EVT = ANALYSIS_OUTPUT_ROOT / "output_EVT"
OUTDIR_LLI.mkdir(parents=True, exist_ok=True)
OUTDIR_EVT.mkdir(parents=True, exist_ok=True)

print("CSV_PATHS:", CSV_PATHS[:3], "..." if len(CSV_PATHS) > 3 else "")
print("OUTDIR_LLI:", OUTDIR_LLI.resolve())
print("OUTDIR_EVT:", OUTDIR_EVT.resolve())


## 1) 共通関数（NaN 対応 / 列選択 / detrend / z-score / 相関）

- **補間はしません**（signal の NaN は NaN のまま）  
- `state` はイベント検出のために **ffill/bfill** で欠損を埋めます（カテゴリ値なので線形補間はしない）
- 相関は「同時に finite な点のみ」で計算し、lag ごとに **重み (= ペア数)** を保持します


In [ ]:
# ============================================================
# 1) Utilities
# ============================================================

def continuous_reindex_no_interp(sub: pd.DataFrame,
                                 frame_col: str = "frame",
                                 fill_state: bool = True,
                                 state_col: str = "state") -> pd.DataFrame:
    """Reindex to continuous integer frames WITHOUT interpolating signals.
    - Numeric signals are kept as NaN where missing.
    - state can be ffill/bfill (categorical).
    """
    if frame_col not in sub.columns:
        raise KeyError(f"'{frame_col}' column not found.")
    sub = sub.sort_values(frame_col).copy()
    frames = sub[frame_col].to_numpy()
    frames = frames[np.isfinite(frames)].astype(int)
    if len(frames) == 0:
        return sub.iloc[0:0].copy()
    fmin, fmax = int(frames.min()), int(frames.max())
    full = pd.DataFrame({frame_col: np.arange(fmin, fmax + 1, dtype=int)})
    out = full.merge(sub, on=frame_col, how="left", sort=True)
    # Fill state if requested
    if fill_state and (state_col in out.columns):
        out[state_col] = out[state_col].ffill().bfill()
    return out


def local_detrend(x: np.ndarray,
                  window: int = 15,
                  method: str = "median",
                  mode: str = "sub") -> np.ndarray:
    """Rolling detrend (median or mean). NaN-safe via pandas rolling.
    mode:
      - 'sub' : x - trend
      - 'div' : x / trend
    """
    s = pd.Series(x.astype(float))
    if method == "median":
        trend = s.rolling(window, center=True, min_periods=max(3, window//5)).median()
    else:
        trend = s.rolling(window, center=True, min_periods=max(3, window//5)).mean()
    trend = trend.to_numpy()
    if mode == "div":
        with np.errstate(divide="ignore", invalid="ignore"):
            y = x / trend
    else:
        y = x - trend
    return y


def zscore_nan(x: np.ndarray) -> np.ndarray:
    x = x.astype(float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd <= 0:
        return x * np.nan
    return (x - mu) / sd


def choose_signal_column(df: pd.DataFrame, prefix: str, source_mode: str) -> str:
    """Pick which column to use as the *base* signal."""
    mean_col = f"{prefix}_mean"
    det_col  = f"{prefix}_detrended"

    if source_mode == "prefer_mean":
        if mean_col in df.columns:
            return mean_col
        if det_col in df.columns:
            return det_col
    else:  # prefer_detrended (Timelapse_mod behavior)
        if det_col in df.columns:
            return det_col
        if mean_col in df.columns:
            return mean_col

    raise KeyError(f"Neither {mean_col} nor {det_col} found in columns.")


def pick_preprocess(subc: pd.DataFrame, prefix: str,
                    config: dict, addon: dict) -> np.ndarray:
    """Pick signal column -> (optional detrend) -> (optional z-score).
    This matches Timelapse_mod philosophy, with SOURCE_MODE switch.
    """
    col = choose_signal_column(subc, prefix, addon["SOURCE_MODE"])
    x = pd.to_numeric(subc[col], errors="coerce").to_numpy(dtype=float)

    # Apply detrend depending on policy
    if config.get("apply_detrend", True):
        mean_col = f"{prefix}_mean"
        det_col  = f"{prefix}_detrended"
        using_mean = (col == mean_col)
        using_det  = (col == det_col)

        if using_mean or addon.get("FORCE_DETREND", False):
            x = local_detrend(
                x,
                window=int(config.get("detrend_window", 15)),
                method=str(config.get("detrend_method", "median")),
                mode=str(config.get("detrend_mode", "sub")),
            )

    if config.get("use_zscore", True):
        x = zscore_nan(x)

    return x


def xcorr_nan_fast(x: np.ndarray, y: np.ndarray,
                   max_lag: int,
                   min_pairs: int = 3):
    """Cross-correlation r(lag) = mean_t x[t] * y[t+lag] over finite pairs.
    Returns:
      lags: [-max_lag..max_lag]
      rho : shape (2*max_lag+1,)
      w   : number of finite pairs at each lag
    Implementation uses np.correlate on NaN->0 values + mask correlation.
    """
    x = x.astype(float)
    y = y.astype(float)
    n = len(x)
    if len(y) != n:
        raise ValueError("x and y must have same length (after reindex).")
    if n == 0:
        lags = np.arange(-max_lag, max_lag+1, dtype=int)
        return lags, np.full_like(lags, np.nan, dtype=float), np.zeros_like(lags, dtype=int)

    x0 = np.nan_to_num(x, nan=0.0)
    y0 = np.nan_to_num(y, nan=0.0)
    mx = np.isfinite(x).astype(float)
    my = np.isfinite(y).astype(float)

    num_full = np.correlate(x0, y0, mode="full")
    cnt_full = np.correlate(mx, my, mode="full")

    center = n - 1
    lags = np.arange(-max_lag, max_lag + 1, dtype=int)
    rho = np.full(lags.shape, np.nan, dtype=float)
    w = np.zeros(lags.shape, dtype=int)

    for i, lag in enumerate(lags):
        idx = center + lag
        if idx < 0 or idx >= len(num_full):
            continue
        cnt = int(cnt_full[idx])
        w[i] = cnt
        if cnt >= min_pairs:
            rho[i] = float(num_full[idx] / cnt)
    return lags, rho, w


def wmean(x: np.ndarray, w: np.ndarray) -> float:
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not np.any(m):
        return np.nan
    return float(np.sum(x[m] * w[m]) / np.sum(w[m]))

from typing import Tuple
def baseline_correct_by_abs_lag_range(lags: np.ndarray, rho: np.ndarray, w: np.ndarray,
                                     base_lag_range: Tuple[int, int]) -> Tuple[np.ndarray, float]:
    """Subtract weighted baseline offset estimated from |lag| in [lo..hi]."""
    lo, hi = base_lag_range
    abs_lag = np.abs(lags)
    m = (abs_lag >= lo) & (abs_lag <= hi)
    off = wmean(rho[m], w[m])
    return rho - off, off


def lead_lag_index(lags: np.ndarray, rho_corr: np.ndarray, w: np.ndarray,
                   window: int = 5) -> float:
    """LLI = mean(rho at negative lags) - mean(rho at positive lags).
    Negative lags correspond to x leading y under our convention.
    Uses weights (pair counts).
    """
    neg = (lags < 0) & (lags >= -window)
    pos = (lags > 0) & (lags <= window)
    neg_mean = wmean(rho_corr[neg], w[neg])
    pos_mean = wmean(rho_corr[pos], w[pos])
    if not np.isfinite(neg_mean) or not np.isfinite(pos_mean):
        return np.nan
    return float(neg_mean - pos_mean)


def bootstrap_ci(x: np.ndarray, stat_fn=np.nanmedian, B: int = 3000, seed: int = 0):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    stats = np.empty(B, float)
    n = len(x)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        stats[b] = stat_fn(x[idx])
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return (float(lo), float(hi))


def bootstrap_mean_curve(curves: np.ndarray, B: int = 2000, seed: int = 0):
    """curves: (n_cells, n_lags). Returns mean and pointwise CI."""
    curves = np.asarray(curves, float)
    n_cells, n_lags = curves.shape
    rng = np.random.default_rng(seed)
    boots = np.empty((B, n_lags), float)
    for b in range(B):
        idx = rng.integers(0, n_cells, size=n_cells)
        boots[b] = np.nanmean(curves[idx], axis=0)
    mean = np.nanmean(curves, axis=0)
    lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
    return mean, lo, hi

from typing import List, Tuple
def time_shift_perm_test(cell_xy: List[Tuple[np.ndarray, np.ndarray]],
                         config: dict, addon: dict,
                         seed: int = 0):
    """Time-shift permutation on y within each cell.
    Returns null distribution of median LLI across cells (length B_PERM).
    """
    rng = np.random.default_rng(seed)
    max_lag = int(config["max_lag"])
    min_pairs = int(config["pair_thresh"])
    base_range = (int(config["base_lag_range"][0]), int(config["base_lag_range"][1]))
    win = int(addon["LLI_WINDOW_FRAMES"])

    B = int(addon["B_PERM"])
    stats = np.full(B, np.nan, float)

    for b in range(B):
        lli_b = []
        for (x, y) in cell_xy:
            n = len(x)
            if n < (2 * max_lag + 3):
                continue
            # choose shift avoiding trivial shifts around 0
            min_shift = max_lag + 1
            max_shift = n - (max_lag + 1)
            if max_shift <= min_shift:
                continue
            shift = int(rng.integers(min_shift, max_shift))
            ysh = np.roll(y, shift)
            lags, rho, w = xcorr_nan_fast(x, ysh, max_lag=max_lag, min_pairs=min_pairs)
            rho_corr, _ = baseline_correct_by_abs_lag_range(lags, rho, w, base_range)
            lli = lead_lag_index(lags, rho_corr, w, window=win)
            if np.isfinite(lli):
                lli_b.append(lli)
        if len(lli_b) > 0:
            stats[b] = float(np.nanmedian(lli_b))
    return stats


def pvalues_from_null(null_stats: np.ndarray, obs: float):
    null_stats = np.asarray(null_stats, float)
    null_stats = null_stats[np.isfinite(null_stats)]
    if len(null_stats) == 0 or not np.isfinite(obs):
        return dict(p_two=np.nan, p_one=np.nan, n_null=int(len(null_stats)))
    # add +1 smoothing
    B = len(null_stats)
    p_two = (np.sum(np.abs(null_stats) >= abs(obs)) + 1) / (B + 1)
    if obs >= 0:
        p_one = (np.sum(null_stats >= obs) + 1) / (B + 1)
    else:
        p_one = (np.sum(null_stats <= obs) + 1) / (B + 1)
    return dict(p_two=float(p_two), p_one=float(p_one), n_null=int(B))


def bh_fdr(pvals: np.ndarray):
    """Benjamini–Hochberg FDR. Returns q-values (same shape)."""
    p = np.asarray(pvals, float)
    q = np.full_like(p, np.nan, dtype=float)
    m = np.isfinite(p)
    if not np.any(m):
        return q
    pv = p[m]
    order = np.argsort(pv)
    pv_sorted = pv[order]
    n = len(pv_sorted)
    q_sorted = pv_sorted * n / (np.arange(1, n+1))
    # enforce monotonicity
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q_sorted = np.clip(q_sorted, 0, 1)
    q_vals = np.empty(n, float)
    q_vals[order] = q_sorted
    q[m] = q_vals
    return q


## 2) LLI 解析（Timelapse_mod データに対して）

### 定義（このノートの LLI）
- per-cell の baseline-corrected cross-correlation `rho_corr(τ)` を使う
- `LLI = mean_{τ=-W..-1} rho_corr(τ) - mean_{τ=1..W} rho_corr(τ)`
  - **LLI > 0** : 負ラグ側の相関が高い → **x が y に先行**（Timelapse_mod の「Neg τ: SNAP leads」と整合）
  - **LLI < 0** : 正ラグ側が優位 → **y が x に先行**

### 統計
- per-cell LLI の **median** を観測統計量とします
- null は **各細胞内で y をランダムに time-shift**（np.roll）して median LLI を再計算（`B_PERM` 回）
- two-sided / one-sided の p-value を出力します

> 「補間しない」方針のため、NaN が多いデータでは perm の null が不安定になる可能性があります。  
> その場合は `EVENT_MIN_FRACTION_FINITE` や `pair_thresh` の調整を検討してください。


In [ ]:
# ============================================================
# 2) Run LLI analysis
# ============================================================

def run_lli_all(csv_paths, config, addon, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)

    max_lag = int(config["max_lag"])
    min_pairs = int(config["pair_thresh"])
    base_range = (int(config["base_lag_range"][0]), int(config["base_lag_range"][1]))
    win = int(addon["LLI_WINDOW_FRAMES"])

    rows = []
    p_two_list = []
    labels = []

    for csv_path, label in csv_paths:
        csv_path = Path(csv_path)
        print(f"\n[LLI] {label}: {csv_path}")
        if not csv_path.exists():
            print("  -> missing file, skip")
            rows.append(dict(label=label, csv_path=str(csv_path), status="missing_file"))
            continue

        df = pd.read_csv(csv_path)
        lli_cells = []
        r0_cells = []
        cell_ids = []
        curves = []
        weights = []
        cell_xy = []

        for cid, sub in df.groupby("cell_id"):
            subc = continuous_reindex_no_interp(sub, fill_state=False)
            if len(subc) < max(config.get("cell_min_points", 8), (2 * max_lag + 3)):
                continue

            try:
                x = pick_preprocess(subc, "ch1", config, addon)  # SNAP/factor
                y = pick_preprocess(subc, "ch2", config, addon)  # MCP
            except Exception:
                continue

            if np.all(~np.isfinite(x)) or np.all(~np.isfinite(y)):
                continue

            lags, rho, w = xcorr_nan_fast(x, y, max_lag=max_lag, min_pairs=min_pairs)
            rho_corr, off = baseline_correct_by_abs_lag_range(lags, rho, w, base_range)

            lli = lead_lag_index(lags, rho_corr, w, window=win)
            if not np.isfinite(lli):
                continue

            # r(0) after baseline correction
            zi = np.where(lags == 0)[0]
            r0 = float(rho_corr[zi[0]]) if len(zi) else np.nan

            lli_cells.append(lli)
            r0_cells.append(r0)
            cell_ids.append(cid)
            curves.append(rho_corr)
            weights.append(w)
            cell_xy.append((x, y))

        if len(lli_cells) == 0:
            print("  -> no valid cells")
            rows.append(dict(label=label, csv_path=str(csv_path), status="no_valid_cells"))
            continue

        lli_cells = np.asarray(lli_cells, float)
        r0_cells = np.asarray(r0_cells, float)
        curves = np.vstack(curves)

        # summary stats
        obs_med = float(np.nanmedian(lli_cells))
        ci_lo, ci_hi = bootstrap_ci(lli_cells, stat_fn=np.nanmedian,
                                   B=int(addon["B_BOOT"]), seed=int(addon["SEED"]))

        # permutation test (median across cells)
        null_med = time_shift_perm_test(cell_xy, config, addon, seed=int(addon["SEED"]))
        pinfo = pvalues_from_null(null_med, obs_med)

        # CI band for mean CC curve (optional)
        mean_curve, lo_curve, hi_curve = bootstrap_mean_curve(
            curves, B=min(2000, int(addon["B_BOOT"])), seed=int(addon["SEED"])
        )

        # Save per-cell table
        out_cells_csv = outdir / f"LLI_cells_{label}.csv"
        pd.DataFrame({
            "cell_id": cell_ids,
            "LLI": lli_cells,
            "r0": r0_cells,
        }).to_csv(out_cells_csv, index=False)

        # Plot (PDF)
        out_pdf = outdir / f"LLI_{label}.pdf"
        t_frames = lags
        dt = config.get("dt_min", None)
        if dt is not None and np.isfinite(dt):
            t = t_frames * float(dt)
            xlab = "Lag τ (min)"
        else:
            t = t_frames
            xlab = "Lag τ (frames)"

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        ax = axes[0]
        ax.fill_between(t, lo_curve, hi_curve, alpha=0.2)
        ax.plot(t, mean_curve, label="SNAP vs MCP (baseline-corrected)")
        ax.axhline(0, color="black", lw=0.8, ls="--")
        ax.axvline(0, color="black", lw=0.8, ls=":")
        ax.set_xlabel(xlab)
        ax.set_ylabel("Cross-correlation (z-scored)")
        ax.set_title(f"Cross-correlation mean ±95% CI\nNeg τ: SNAP leads")
        ax.grid(axis="y", linestyle=":", alpha=0.6)
        ax.legend()

        ax = axes[1]
        null_med_f = null_med[np.isfinite(null_med)]
        ax.hist(null_med_f, bins=30, alpha=0.7)
        ax.axvline(obs_med, color="black", lw=1.5, label="Observed median LLI")
        ax.axvline(0, color="black", lw=0.8, ls="--")
        ax.set_xlabel("Median LLI (null by time-shift)")
        ax.set_ylabel("Count")
        ax.set_title("Time-shift permutation null")
        ax.grid(axis="y", linestyle=":", alpha=0.6)
        ax.legend()

        txt = (
            f"N_cells={len(lli_cells)}\n"
            f"LLI_median={obs_med:.4f}\n"
            f"95%CI=[{ci_lo:.4f}, {ci_hi:.4f}]\n"
            f"perm p(two)={pinfo['p_two']:.4g}\n"
            f"perm p(one)={pinfo['p_one']:.4g}"
        )
        fig.suptitle(f"{label}\n{txt}", y=1.05)
        fig.tight_layout()
        fig.savefig(out_pdf, bbox_inches="tight")
        plt.close(fig)

        print("  saved:", out_pdf)
        print("  saved:", out_cells_csv)

        row = dict(
            label=label,
            csv_path=str(csv_path),
            status="ok",
            N_cells=int(len(lli_cells)),
            LLI_window_frames=int(win),
            LLI_median=float(obs_med),
            LLI_ci_low=float(ci_lo),
            LLI_ci_high=float(ci_hi),
            perm_B=int(addon["B_PERM"]),
            perm_p_two=float(pinfo["p_two"]),
            perm_p_one=float(pinfo["p_one"]),
            dt_min=float(dt) if (dt is not None and np.isfinite(dt)) else np.nan,
            source_mode=str(addon["SOURCE_MODE"]),
            force_detrend=int(addon["FORCE_DETREND"]),
        )
        rows.append(row)
        labels.append(label)
        p_two_list.append(row["perm_p_two"])

    # FDR across datasets (two-sided)
    p_two_arr = np.asarray(p_two_list, float)
    q_two = bh_fdr(p_two_arr)
    qi = 0
    for r in rows:
        if r.get("status") == "ok":
            r["perm_q_two_BH"] = float(q_two[qi])
            qi += 1
        else:
            r["perm_q_two_BH"] = np.nan

    out_summary = outdir / "LLI_summary_mod.csv"
    pd.DataFrame(rows).to_csv(out_summary, index=False)
    print("\n[LLI DONE] wrote:", out_summary.resolve())
    return pd.DataFrame(rows)

lli_summary_df = run_lli_all(CSV_PATHS, CONFIG, ADDON, OUTDIR_LLI)
lli_summary_df.head()


## 3) Event-aligned 解析（OFF→ON / ON→OFF）

`state` の遷移を使って、SNAP（`ch1`）をアラインします（Timelapse_original.pdf の考え方を踏襲）。

### run-length filter
短い ON/OFF の揺れをイベントと誤認しないよう、  
- `MIN_ON_LEN`（frames）連続で ON
- `MIN_OFF_LEN`（frames）連続で OFF  
を満たす遷移のみを採用します。

### 前後差（pre/post metric）
各細胞の event-aligned 平均トレース `trace(rel)` に対して

- `D = mean(pre) - mean(post)`（Timelapse_original と同じ符号）
  - OFF→ON の場合：**D < 0** なら「イベント後に signal が増える」  
  - ON→OFF の場合：**D > 0** なら「イベント後に signal が減る」

統計：
- per-cell の D について **median と bootstrap CI**
- Wilcoxon signed-rank（`scipy` があれば）で D が 0 からズレるかを評価


In [ ]:
# ============================================================
# 3) Event-aligned analysis
# ============================================================

def extract_onsets_runlen(subc: pd.DataFrame,
                          on_label: str = "ON",
                          state_col: str = "state",
                          min_on_len: int = 5,
                          min_off_len: int = 5,
                          t0: str = "first_on"):
    """Return frame indices (0..len-1) of OFF->ON transitions satisfying run-length.
    t0: 'first_on' uses the first ON frame index as the event.
    """
    st = subc[state_col].astype(str).to_numpy()
    is_on = (st == on_label)
    idx = np.where(np.diff(is_on.astype(int)) == 1)[0] + 1  # OFF->ON start index
    out = []
    for i in idx:
        # check preceding OFF run length
        off_ok = True
        if min_off_len > 0:
            lo = max(0, i - min_off_len)
            off_ok = np.all(~is_on[lo:i])
        # check following ON run length
        on_ok = True
        if min_on_len > 0:
            hi = min(len(is_on), i + min_on_len)
            on_ok = np.all(is_on[i:hi])
        if off_ok and on_ok:
            out.append(int(i))
    # optionally map to first_on frame (already i)
    return np.asarray(out, dtype=int)


def extract_offsets_runlen(subc: pd.DataFrame,
                           on_label: str = "ON",
                           state_col: str = "state",
                           min_on_len: int = 5,
                           min_off_len: int = 5,
                           t0: str = "first_off"):
    """Return frame indices of ON->OFF transitions satisfying run-length.
    t0: 'first_off' uses the first OFF frame index as the event.
    """
    st = subc[state_col].astype(str).to_numpy()
    is_on = (st == on_label)
    idx = np.where(np.diff(is_on.astype(int)) == -1)[0] + 1  # ON->OFF start index (first OFF)
    out = []
    for i in idx:
        # preceding ON run length
        on_ok = True
        if min_on_len > 0:
            lo = max(0, i - min_on_len)
            on_ok = np.all(is_on[lo:i])
        # following OFF run length
        off_ok = True
        if min_off_len > 0:
            hi = min(len(is_on), i + min_off_len)
            off_ok = np.all(~is_on[i:hi])
        if on_ok and off_ok:
            out.append(int(i))
    return np.asarray(out, dtype=int)


def event_aligned_trace(x: np.ndarray, event_idx: np.ndarray,
                        preF: int, postF: int,
                        min_fraction_finite: float = 0.7):
    """Average x around event indices.
    Returns (trace_mean, relF). If none, returns (None, relF).
    """
    x = np.asarray(x, float)
    relF = np.arange(-preF, postF + 1, dtype=int)
    segs = []
    for i in event_idx:
        lo = i - preF
        hi = i + postF
        if lo < 0 or hi >= len(x):
            continue
        seg = x[lo:hi+1]
        if np.isfinite(seg).mean() < min_fraction_finite:
            continue
        segs.append(seg)
    if len(segs) == 0:
        return None, relF
    segs = np.vstack(segs)
    return np.nanmean(segs, axis=0), relF


def pre_post_metric(trace: np.ndarray, relF: np.ndarray,
                    pre_win: Tuple[int, int], post_win: Tuple[int, int]) -> float:
    """D = mean(pre) - mean(post) using relF windows (in frames)."""
    pre_mask = (relF >= pre_win[0]) & (relF <= pre_win[1])
    post_mask = (relF >= post_win[0]) & (relF <= post_win[1])
    pre = np.nanmean(trace[pre_mask]) if np.any(pre_mask) else np.nan
    post = np.nanmean(trace[post_mask]) if np.any(post_mask) else np.nan
    if not np.isfinite(pre) or not np.isfinite(post):
        return np.nan
    return float(pre - post)


def summarize_transition(cell_dict: dict,
                         relF: np.ndarray,
                         pre_win: Tuple[int, int],
                         post_win: Tuple[int, int],
                         B: int, seed: int):
    """Return mean trace, CI, and per-cell D stats."""
    if len(cell_dict) == 0:
        return None

    cell_means = np.vstack(list(cell_dict.values()))
    n_cells, nT = cell_means.shape

    mean_tr, lo, hi = bootstrap_mean_curve(cell_means, B=min(2000, B), seed=seed)

    # per-cell D
    D = np.array([pre_post_metric(cell_means[i], relF, pre_win, post_win) for i in range(n_cells)], float)
    D = D[np.isfinite(D)]
    if len(D) == 0:
        D_med = np.nan
        D_ci = (np.nan, np.nan)
        pD = np.nan
    else:
        D_med = float(np.nanmedian(D))
        D_ci = bootstrap_ci(D, stat_fn=np.nanmedian, B=B, seed=seed)
        if wilcoxon is not None:
            try:
                pD = float(wilcoxon(D, alternative="two-sided").pvalue)
            except Exception:
                pD = np.nan
        else:
            pD = np.nan

    return dict(
        n_cells=int(n_cells),
        mean=mean_tr, lo=lo, hi=hi,
        D_values=D,
        D_median=float(D_med),
        D_ci_low=float(D_ci[0]),
        D_ci_high=float(D_ci[1]),
        p_wilcoxon=float(pD) if np.isfinite(pD) else np.nan,
    )


def run_event_align_all(csv_paths, config, addon, outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)

    state_col = addon.get("STATE_COL", "state")
    on_label = addon.get("ON_LABEL", "ON")

    preF = int(addon["EVENT_WINDOW_FRAMES"])
    postF = int(addon["EVENT_WINDOW_FRAMES"])
    relF = np.arange(-preF, postF + 1, dtype=int)

    pre_win = tuple(addon["PRE_WIN_FRAMES"])
    post_win = tuple(addon["POST_WIN_FRAMES"])

    rows = []

    for csv_path, label in csv_paths:
        csv_path = Path(csv_path)
        print(f"\n[EVT] {label}: {csv_path}")
        if not csv_path.exists():
            print("  -> missing file, skip")
            rows.append(dict(label=label, csv_path=str(csv_path), status="missing_file"))
            continue

        df = pd.read_csv(csv_path)
        if state_col not in df.columns:
            print(f"  -> '{state_col}' not found. skip event-aligned for this dataset.")
            rows.append(dict(label=label, csv_path=str(csv_path), status="no_state_column"))
            continue

        cell_on = {}   # OFF->ON
        cell_off = {}  # ON->OFF

        for cid, sub in df.groupby("cell_id"):
            subc = continuous_reindex_no_interp(sub, fill_state=True, state_col=state_col)
            if len(subc) < (preF + postF + 5):
                continue

            try:
                x = pick_preprocess(subc, "ch1", config, addon)  # SNAP/factor
            except Exception:
                continue

            if np.all(~np.isfinite(x)):
                continue

            onsets = extract_onsets_runlen(
                subc,
                on_label=on_label,
                state_col=state_col,
                min_on_len=int(addon["MIN_ON_LEN"]),
                min_off_len=int(addon["MIN_OFF_LEN"]),
                t0="first_on",
            )
            offsets = extract_offsets_runlen(
                subc,
                on_label=on_label,
                state_col=state_col,
                min_on_len=int(addon["MIN_ON_LEN"]),
                min_off_len=int(addon["MIN_OFF_LEN"]),
                t0="first_off",
            )

            if len(onsets) > 0:
                tr, _ = event_aligned_trace(
                    x, onsets, preF, postF,
                    min_fraction_finite=float(addon["EVENT_MIN_FRACTION_FINITE"])
                )
                if tr is not None:
                    cell_on[cid] = tr

            if len(offsets) > 0:
                tr, _ = event_aligned_trace(
                    x, offsets, preF, postF,
                    min_fraction_finite=float(addon["EVENT_MIN_FRACTION_FINITE"])
                )
                if tr is not None:
                    cell_off[cid] = tr

        # summarize both transitions
        B = int(addon["B_BOOT"])
        seed = int(addon["SEED"])
        sum_on = summarize_transition(cell_on, relF, pre_win, post_win, B=B, seed=seed)
        sum_off = summarize_transition(cell_off, relF, pre_win, post_win, B=B, seed=seed)

        # plot
        out_pdf = outdir / f"event_aligned_{label}.pdf"

        dt = config.get("dt_min", None)
        if dt is not None and np.isfinite(dt):
            t = relF * float(dt)
            xlab = "Time from transition (min)"
        else:
            t = relF
            xlab = "Time from transition (frames)"

        fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

        # OFF->ON
        ax = axes[0]
        ax.axvline(0, color="black", lw=0.8, ls=":")
        ax.axhline(0, color="black", lw=0.8, ls="--")
        if sum_on is not None:
            ax.fill_between(t, sum_on["lo"], sum_on["hi"], alpha=0.2)
            ax.plot(t, sum_on["mean"], label=f"N={sum_on['n_cells']}")
            title = f"OFF→ON (D median={sum_on['D_median']:.4f}, p={sum_on['p_wilcoxon']:.3g})"
        else:
            title = "OFF→ON (no valid cells)"
        ax.set_title(title)
        ax.set_xlabel(xlab)
        ax.set_ylabel("SNAP (z-scored)")
        ax.grid(axis="y", linestyle=":", alpha=0.6)
        ax.legend()

        # ON->OFF
        ax = axes[1]
        ax.axvline(0, color="black", lw=0.8, ls=":")
        ax.axhline(0, color="black", lw=0.8, ls="--")
        if sum_off is not None:
            ax.fill_between(t, sum_off["lo"], sum_off["hi"], alpha=0.2)
            ax.plot(t, sum_off["mean"], label=f"N={sum_off['n_cells']}")
            title = f"ON→OFF (D median={sum_off['D_median']:.4f}, p={sum_off['p_wilcoxon']:.3g})"
        else:
            title = "ON→OFF (no valid cells)"
        ax.set_title(title)
        ax.set_xlabel(xlab)
        ax.grid(axis="y", linestyle=":", alpha=0.6)
        ax.legend()

        fig.suptitle(label)
        fig.tight_layout()
        fig.savefig(out_pdf, bbox_inches="tight")
        plt.close(fig)

        print("  saved:", out_pdf)

        # save per-cell D tables
        if sum_on is not None:
            pd.DataFrame({"D_pre_minus_post": sum_on["D_values"]}).to_csv(outdir / f"eventD_OFFtoON_{label}.csv", index=False)
        if sum_off is not None:
            pd.DataFrame({"D_pre_minus_post": sum_off["D_values"]}).to_csv(outdir / f"eventD_ONtoOFF_{label}.csv", index=False)

        # record summary rows
        row_base = dict(
            label=label,
            csv_path=str(csv_path),
            dt_min=float(dt) if (dt is not None and np.isfinite(dt)) else np.nan,
            event_window_frames=int(preF),
            pre_win_frames=str(pre_win),
            post_win_frames=str(post_win),
            source_mode=str(addon["SOURCE_MODE"]),
            force_detrend=int(addon["FORCE_DETREND"]),
        )

        if sum_on is not None:
            rows.append({**row_base,
                         "transition": "OFF_to_ON",
                         "status": "ok",
                         "n_cells": int(sum_on["n_cells"]),
                         "D_median": float(sum_on["D_median"]),
                         "D_ci_low": float(sum_on["D_ci_low"]),
                         "D_ci_high": float(sum_on["D_ci_high"]),
                         "p_wilcoxon": float(sum_on["p_wilcoxon"]),
                         })
        else:
            rows.append({**row_base, "transition": "OFF_to_ON", "status": "no_valid_cells"})

        if sum_off is not None:
            rows.append({**row_base,
                         "transition": "ON_to_OFF",
                         "status": "ok",
                         "n_cells": int(sum_off["n_cells"]),
                         "D_median": float(sum_off["D_median"]),
                         "D_ci_low": float(sum_off["D_ci_low"]),
                         "D_ci_high": float(sum_off["D_ci_high"]),
                         "p_wilcoxon": float(sum_off["p_wilcoxon"]),
                         })
        else:
            rows.append({**row_base, "transition": "ON_to_OFF", "status": "no_valid_cells"})

    out_summary = outdir / "event_aligned_summary_mod.csv"
    pd.DataFrame(rows).to_csv(out_summary, index=False)
    print("\n[EVT DONE] wrote:", out_summary.resolve())
    return pd.DataFrame(rows)

evt_summary_df = run_event_align_all(CSV_PATHS, CONFIG, ADDON, OUTDIR_EVT)
evt_summary_df.head()


# Timelapse_last 追加: 最終図（LLI forest + cross-corr/LLI evidence）作成

このノートは **Timelapse_last** の計算方法（NaN 対応 cross-corr + LLI + time-shift permutation）をそのまま使い、  
論文図として使いやすい **2種類の PDF 図** をまとめて出力します。

1. **Forest plot**（各ファクターの LLI median ± bootstrap 95%CI と、time-shift permutation の p/q）
2. **Evidence figure**（選択したファクターについて、左：mean cross-corr ±95%CI、右：LLI null histogram + 観測値）

> ⚠️ 前提  
> この追加セルは、`Timelapse_last` のノートブックで **LLI 解析まで実行済み**（少なくとも `CONFIG`, `ADDON`, `LLI_summary_mod.csv` が作成済み）であることを想定しています。  
> （`run_lli_all(...)` 相当が実行済みで、`output_LLI/LLI_summary_mod.csv` が存在する状態）



In [ ]:
# ============================================================
# 0) 図の設定（ここだけ編集すればOK）
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.gridspec as gridspec
import warnings

# ---- 図の見た目・出力 ----
FIGCFG = {
    # Font (requested: Arial)
    "font_family": "Arial",

    # Font sizes (自由に変更可)
    "base_fontsize": 12,
    "title_fontsize": 12,
    "axes_title_fontsize": 12,
    "label_fontsize": 12,
    "tick_fontsize": 12,
    "legend_fontsize": 12,

    # 出力先
    # None の場合: ADDON["OUTDIR_LLI"]/final_figs に出力
    "outdir": None,
    "forest_pdf": "Fig_LLI_forest.pdf",
    "evidence_pdf": "Fig_LLI_crosscorr_evidence.pdf",

    # ---- どのファクターを図にするか ----
    # None -> LLI_summary_mod.csv の status==ok を全部
    "forest_factors": ["SOX2","BRD4", "SIN3A", "HDAC3"],  #None,

    # Evidence（2列×N行）に載せるファクター（順序もここで指定）
    "evidence_factors": ["SOX2","BRD4", "SIN3A", "HDAC3"],

    # ---- レイアウト（自由に変更可）----
    # Forest
    "forest_width": 5.0,         # inch
    "forest_row_height": 0.65,    # inch / factor
    "forest_xpad": 0.2,          # xlim 左右の余白（全幅に対する割合）

    # Evidence
    "evidence_fig_width": 14.0,   # inch
    "evidence_row_height": 4.0,   # inch / factor row
    "evidence_rows_per_page": 4,  # 多数ファクターの場合に何行ごとに改ページするか
    "evidence_wspace": 0.25,
    "evidence_hspace": 0.55,

    # ---- 描画 ----
    "null_bins": 30,              # null histogram bins
    "shade_lli_window": True,     # cross-corr の LLI window を薄灰で表示
    "show_inline": True,          # Notebook 上にも図を表示する
}

# ---- Matplotlib global settings: Arial + vector PDF ----
mpl.rcParams.update({
    "font.family": FIGCFG["font_family"],
    "pdf.fonttype": 42,  # TrueType for Illustrator friendliness
    "ps.fonttype": 42,
    "font.size": FIGCFG["base_fontsize"],
    "axes.titlesize": FIGCFG["axes_title_fontsize"],
    "axes.labelsize": FIGCFG["label_fontsize"],
    "xtick.labelsize": FIGCFG["tick_fontsize"],
    "ytick.labelsize": FIGCFG["tick_fontsize"],
    "legend.fontsize": FIGCFG["legend_fontsize"],
})

# Arial が無い環境だと別フォントにフォールバックします（警告だけ出す）
try:
    import matplotlib.font_manager as fm
    has_arial = any(("Arial" in f.name) for f in fm.fontManager.ttflist)
    if not has_arial:
        warnings.warn("Arial フォントが見つかりません。環境によっては別フォントに置換されます。")
except Exception:
    pass

print("Matplotlib font:", mpl.rcParams["font.family"])


## 1) 前提チェック（Timelapse_last を実行済みか）

この追加セルは、Timelapse_last の中で定義された以下を使います：

- `CONFIG`, `ADDON`
- 前処理/計算関数: `continuous_reindex_no_interp`, `pick_preprocess`, `xcorr_nan_fast`, `baseline_correct_by_abs_lag_range`, `lead_lag_index`, `time_shift_perm_test`
- `output_LLI/LLI_summary_mod.csv`（run_lli_all が作る summary）

未実行の場合は、先に Timelapse_last の LLI セクションまでを実行してから進めてください。


In [ ]:
# ============================================================
# 1) 前提チェック
# ============================================================

REQUIRED = [
    "CONFIG", "ADDON",
    "continuous_reindex_no_interp",
    "pick_preprocess",
    "xcorr_nan_fast",
    "baseline_correct_by_abs_lag_range",
    "lead_lag_index",
    "time_shift_perm_test",
]
missing = [name for name in REQUIRED if name not in globals()]
if missing:
    raise RuntimeError(
        "必要な変数/関数が見つかりません。\n"
        "このノートは Timelapse_last の LLI 解析まで実行済みであることを前提にしています。\n"
        f"Missing: {missing}"
    )

# output_LLI フォルダ
OUTDIR_LLI = globals().get(
    "OUTDIR_LLI", ANALYSIS_OUTPUT_ROOT / "output_LLI"
)
print("OUTDIR_LLI:", OUTDIR_LLI.resolve())


## 2) LLI summary を読み込む

Timelapse_last では LLI 解析の結果が `output_LLI/LLI_summary_mod.csv` に保存されています。  
ここではそれを読み込み、以後の作図で使いやすい列名に整形します。

（Timelapse_last の run_lli_all が `LLI_summary_mod.csv` を出力する流れに合わせています。）

In [ ]:
# ============================================================
# 2) Load LLI_summary_mod.csv
# ============================================================

summary_path = OUTDIR_LLI / "LLI_summary_mod.csv"
if not summary_path.exists():
    raise FileNotFoundError(f"LLI summary not found: {summary_path}")

lli_summary_raw = pd.read_csv(summary_path)

# status==ok だけ使う
lli_summary_ok = lli_summary_raw.copy()
if "status" in lli_summary_ok.columns:
    lli_summary_ok = lli_summary_ok[lli_summary_ok["status"].astype(str) == "ok"].copy()

# 作図用に列名を統一
rename_map = {
    "label": "factor",
    "N_cells": "n_cells",
    "LLI_median": "lli_median",
    "LLI_ci_low": "lli_ci_low",
    "LLI_ci_high": "lli_ci_high",
    "perm_p_two": "p_perm_two",
    "perm_q_two_BH": "q_perm_two",
}
for k, v in rename_map.items():
    if k in lli_summary_ok.columns and v not in lli_summary_ok.columns:
        lli_summary_ok = lli_summary_ok.rename(columns={k: v})

need_cols = ["factor", "csv_path", "n_cells", "lli_median", "lli_ci_low", "lli_ci_high", "p_perm_two", "q_perm_two"]
missing_cols = [c for c in need_cols if c not in lli_summary_ok.columns]
if missing_cols:
    raise RuntimeError(f"LLI summary に必要列がありません: {missing_cols}\ncolumns={list(lli_summary_ok.columns)}")

lli_summary_ok = lli_summary_ok.sort_values("p_perm_two", kind="mergesort").reset_index(drop=True)

display(lli_summary_ok[need_cols].head(10))
print("n_factors(ok):", len(lli_summary_ok))


## 3) ファクター選択（Forest と Evidence で別々に設定）

- `FIGCFG["forest_factors"]`  
  - `None` → summary の `status==ok` を全て表示  
  - `["BRD4","HDAC3", ...]` のようにリスト指定すると、その順序で表示

- `FIGCFG["evidence_factors"]`  
  - Evidence 図（cross-corr + null hist）の対象  
  - **順序がそのまま図の上から下になります**



In [ ]:
# ============================================================
# 3) Resolve factor lists
# ============================================================

available_factors = lli_summary_ok["factor"].astype(str).tolist()

def resolve_factor_list(requested, available):
    if requested is None:
        return list(available)
    requested = [str(x) for x in requested]
    missing = [x for x in requested if x not in set(available)]
    if missing:
        warnings.warn(f"指定した factor が summary にありません（無視します）: {missing}")
    return [x for x in requested if x in set(available)]

forest_factors = resolve_factor_list(FIGCFG["forest_factors"], available_factors)
evidence_factors = resolve_factor_list(FIGCFG["evidence_factors"], available_factors)

print("Forest factors:", forest_factors)
print("Evidence factors:", evidence_factors)


## 4) Evidence 用の「事前計算」：mean cross-corr CI と null 分布をキャッシュする

Forest plot は `LLI_summary_mod.csv` だけで作れますが、Evidence 図（右側 histogram）には **time-shift null の実データ配列**が必要です。

Timelapse_last の `run_lli_all` では null 分布を内部で計算して図を保存しますが、  
**null 分布の配列そのものは CSV に保存していません**（図だけ）。

そこでこのセルでは、選択した `evidence_factors` について

- per-cell の baseline-corrected cross-corr 曲線（rho_corr）
- mean 曲線と bootstrap 95% CI
- time-shift permutation による null（median LLI の分布）

を再計算し、`*.npz` としてキャッシュします。

> 一度キャッシュしておけば、図サイズ/フォントを変えても再計算せず高速に PDF を作り直せます。


In [ ]:
# ============================================================
# 4) Precompute dataset objects (cache to npz)
# ============================================================

import re

# cache dir
FIG_OUTDIR = Path(FIGCFG["outdir"]) if FIGCFG["outdir"] else (OUTDIR_LLI / "final_figs")
FIG_OUTDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = FIG_OUTDIR / "_precompute_npz"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _safe_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s))

def get_row_for_factor(summary_df: pd.DataFrame, factor: str) -> pd.Series:
    sub = summary_df[summary_df["factor"].astype(str) == str(factor)]
    if len(sub) == 0:
        raise KeyError(f"factor not found in summary: {factor}")
    # 一応、p が小さい方を優先
    sub = sub.sort_values("p_perm_two", kind="mergesort")
    return sub.iloc[0]

def bootstrap_mean_curve(curves: np.ndarray, B: int, seed: int):
    """curves: (n_cells, n_lags) with NaNs allowed"""
    rng = np.random.default_rng(seed)
    n_cells, n_lags = curves.shape
    boots = np.empty((B, n_lags), float)
    for b in range(B):
        idx = rng.integers(0, n_cells, size=n_cells)
        boots[b] = np.nanmean(curves[idx], axis=0)
    mean_curve = np.nanmean(curves, axis=0)
    lo, hi = np.nanpercentile(boots, [2.5, 97.5], axis=0)
    return mean_curve, lo, hi

def precompute_factor_dataset(factor: str, summary_row: pd.Series, *, recompute: bool = False, verbose: bool = True):
    cache_path = CACHE_DIR / f"precompute_{_safe_name(factor)}.npz"
    if cache_path.exists() and (not recompute):
        z = np.load(cache_path, allow_pickle=True)
        ds = {k: z[k] for k in z.files}
        # restore scalars
        for k in ["dt_min", "n_cells", "win_frames", "max_lag"]:
            if k in ds:
                ds[k] = float(ds[k]) if k != "n_cells" else int(ds[k])
        ds["factor"] = factor
        if verbose:
            print(f"[cache] {factor}: {cache_path.name}")
        return ds

    csv_path = Path(str(summary_row["csv_path"]))
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found for {factor}: {csv_path}")

    # per-factor override (if summary keeps them)
    cfg = dict(CONFIG)
    addon = dict(ADDON)
    if "source_mode" in summary_row.index:
        addon["SOURCE_MODE"] = str(summary_row["source_mode"])
    if "force_detrend" in summary_row.index:
        try:
            addon["FORCE_DETREND"] = bool(int(summary_row["force_detrend"]))
        except Exception:
            addon["FORCE_DETREND"] = bool(summary_row["force_detrend"])

    # parameters (follow Timelapse_last)
    max_lag = int(cfg["max_lag"])
    min_pairs = int(cfg["pair_thresh"])
    base_range = (int(cfg["base_lag_range"][0]), int(cfg["base_lag_range"][1]))
    win = int(summary_row.get("LLI_window_frames", addon.get("LLI_WINDOW_FRAMES", 5)))
    B_BOOT = int(addon.get("B_BOOT", int(cfg.get("boot_n", 3000))))
    seed = int(addon.get("SEED", 12345))

    # dt (for plotting)
    dt_min = summary_row.get("dt_min", np.nan)
    try:
        dt_min = float(dt_min)
    except Exception:
        dt_min = np.nan
    if not np.isfinite(dt_min):
        dt_min = float(cfg.get("dt_min", np.nan))

    # load + per-cell processing
    df = pd.read_csv(csv_path)

    curves = []
    cell_xy = []
    lli_vals = []

    # Timelapse_last は cell_id / frame を前提としている
    group_col = cfg.get("cell_id_col", "cell_id")
    if group_col not in df.columns:
        group_col = "cell_id"
    if group_col not in df.columns:
        raise RuntimeError(f"cell id column not found: tried '{cfg.get('cell_id_col','cell_id')}' and 'cell_id'. cols={list(df.columns)}")

    for cid, sub in df.groupby(group_col):
        # fill missing frames with NaN, no interpolation (Timelapse_last 方針)
        subc = continuous_reindex_no_interp(sub, fill_state=False)

        if len(subc) < max(cfg.get("cell_min_points", 8), (2 * max_lag + 3)):
            continue

        try:
            x = pick_preprocess(subc, "ch1", config=cfg, addon=addon)
            y = pick_preprocess(subc, "ch2", config=cfg, addon=addon)

        except Exception:
            continue

        lags, rho, w = xcorr_nan_fast(x, y, max_lag=max_lag, min_pairs=min_pairs)
        rho_corr, _ = baseline_correct_by_abs_lag_range(lags, rho, w, base_range)
        lli = lead_lag_index(lags, rho_corr, w, window=win)

        if not np.isfinite(lli):
            continue

        curves.append(rho_corr)
        cell_xy.append((x, y))
        lli_vals.append(float(lli))

    if len(curves) == 0:
        raise RuntimeError(f"No valid cells for {factor}. Check preprocessing/QC thresholds.")

    curves = np.vstack(curves)
    lli_vals = np.asarray(lli_vals, float)

    # mean curve + CI (bootstrap across cells)
    mean_curve, ci_lo, ci_hi = bootstrap_mean_curve(curves, B=B_BOOT, seed=seed)

    # time-shift null (median LLI)
    null_med = time_shift_perm_test(cell_xy, config=cfg, addon=addon)

    obs_med = float(np.nanmedian(lli_vals))

    # axis in minutes
    if np.isfinite(dt_min):
        t = lags.astype(float) * float(dt_min)
        xlab = "Lag τ (min)"
    else:
        t = lags.astype(float)
        xlab = "Lag τ (frames)"

    ds = dict(
        factor=factor,
        csv_path=str(csv_path),
        lags=lags.astype(int),
        t=t.astype(float),
        xlab=str(xlab),
        dt_min=float(dt_min) if np.isfinite(dt_min) else np.nan,
        max_lag=int(max_lag),
        win_frames=int(win),
        mean_curve=mean_curve.astype(float),
        ci_lo=ci_lo.astype(float),
        ci_hi=ci_hi.astype(float),
        null_med=null_med.astype(float),
        obs_med=float(obs_med),
        n_cells=int(curves.shape[0]),
        # store baseline settings for traceability
        base_range=str(base_range),
        source_mode=str(addon.get("SOURCE_MODE")),
        force_detrend=int(bool(addon.get("FORCE_DETREND", False))),
    )

    np.savez_compressed(cache_path, **ds)
    if verbose:
        print(f"[computed] {factor}: n_cells={ds['n_cells']} -> cached {cache_path.name}")
    return ds

# ---- actually precompute selected evidence factors ----
datasets = {}
for fac in evidence_factors:
    row = get_row_for_factor(lli_summary_ok, fac)
    datasets[fac] = precompute_factor_dataset(fac, row, recompute=False, verbose=True)

print("precomputed:", list(datasets.keys()))


## 5) 作図関数（Forest / Evidence）

Timelapse_original の「図のレイアウト」だけを参考にしつつ、  
**計算値自体は Timelapse_last の結果（LLI_summary と precompute datasets）を使う**構成にしています。

- Forest: summary の `lli_median` と `lli_ci_low/high` をそのまま表示
- Evidence: datasets に入っている `mean_curve ± CI` と `null_med` を表示



In [ ]:
# ============================================================
# 5) Plot helpers
# ============================================================

LLI_SIGN = -1 # -: MCP leads, +: SNAP leads

def _fmt_float(x, nd=4):
    try:
        x = float(x)
    except Exception:
        return "nan"
    if not np.isfinite(x):
        return "nan"
    # show 0.0005 as 0.0005 (like your example)
    if 0 < abs(x) < 0.001:
        return f"{x:.4f}"
    if abs(x) < 0.01:
        return f"{x:.3f}"
    if abs(x) < 1:
        return f"{x:.3g}"
    return f"{x:.3g}"

def plot_crosscorr_panel(ax, ds, *, title=None, annotate=True, lli_row=None):
    t = ds["t"]
    ax.fill_between(t, ds["ci_lo"], ds["ci_hi"], alpha=0.2)
    ax.plot(t, ds["mean_curve"], lw=2.0)
    ax.axvline(0, color="black", lw=1.0, ls=":")
    ax.axhline(0, color="black", lw=1.0, ls="--")

    # shade LLI window
    if FIGCFG.get("shade_lli_window", True) and np.isfinite(ds.get("dt_min", np.nan)):
        dt = float(ds["dt_min"])
        w = int(ds["win_frames"])
        # window = lags [-w..-1] and [1..w]
        ax.axvspan(-w * dt, -1 * dt, color="0.92", zorder=0)
        ax.axvspan( 1 * dt,  w * dt, color="0.92", zorder=0)

    if title is None:
        title = f"{ds['factor']}: mean cross-corr ±95% CI"
    ax.set_title(title)
    ax.set_xlabel(ds.get("xlab", "Lag τ"))
    ax.set_ylabel("Cross-corr (baseline-corrected)")
    ax.grid(axis="y", linestyle=":", alpha=0.6)

    if annotate:
        if lli_row is None:
            lli_row = get_row_for_factor(lli_summary_ok, ds["factor"])
        txt = (
            f"N={int(lli_row['n_cells'])}\n"
            f"LLI_med={LLI_SIGN * float(lli_row['lli_median']):.4g}\n"
            f"perm p={_fmt_float(lli_row['p_perm_two'])}\n"
            f"q={_fmt_float(lli_row['q_perm_two'])}"
        )
        ax.text(
            0.02, 0.98, txt,
            transform=ax.transAxes,
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.7", alpha=0.95),
        )

def plot_null_hist_panel(ax, ds, *, title=None, show_legend=False):
    null_med = LLI_SIGN * ds["null_med"]
    obs = LLI_SIGN * float(ds["obs_med"])

    ax.hist(null_med, bins=int(FIGCFG.get("null_bins", 30)), color="0.85", edgecolor="0.5")
    ax.axvline(0, color="black", lw=1.2, ls="--", alpha=0.7)
    ax.axvline(obs, color="black", lw=3.0)

    if title is None:
        title = f"{ds['factor']}: time-shift null vs observed"
    ax.set_title(title)
    ax.set_xlabel("Median LLI (time-shift null)")
    ax.set_ylabel("Count")
    ax.grid(axis="y", linestyle=":", alpha=0.3)

    if show_legend:
        ax.plot([], [], color="black", lw=3.0, label="Observed median LLI")
        ax.plot([], [], color="black", lw=1.2, ls="--", alpha=0.7, label="Null center (0)")
        ax.legend(loc="best")


## 6) PDF 出力（Forest / Evidence）

- Forest: `FIGCFG["forest_pdf"]`
- Evidence: `FIGCFG["evidence_pdf"]`  
  `FIGCFG["evidence_rows_per_page"]` を超える数のファクターを指定した場合は自動で改ページします。


In [ ]:
# ============================================================
# 6) Export functions
# ============================================================

def export_forest_pdf(pdf_path: Path, factors):
    df = lli_summary_ok.set_index("factor").loc[factors].reset_index()

    n = len(df)
    fig_h = max(2.0, FIGCFG["forest_row_height"] * n + 1.0)
    fig_w = FIGCFG["forest_width"]

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    # y positions: top to bottom in given order
    y = np.arange(n)
    ax.invert_yaxis()

    #x = df["lli_median"].astype(float).to_numpy()
    x = LLI_SIGN * df["lli_median"].astype(float).to_numpy()
    #lo = df["lli_ci_low"].astype(float).to_numpy()
    lo0 = df["lli_ci_low"].astype(float).to_numpy()
    hi0 = df["lli_ci_high"].astype(float).to_numpy()
    lo = LLI_SIGN * hi0
    #hi = df["lli_ci_high"].astype(float).to_numpy()
    hi = LLI_SIGN * lo0

    # error bar (median ± CI)
    xerr = np.vstack([x - lo, hi - x])
    ax.errorbar(x, y, xerr=xerr, fmt="o", color="black", ecolor="black", capsize=3)

    ax.axvline(0, color="black", lw=1.2, ls="--")
    ax.set_yticks(y)
    ax.set_yticklabels(df["factor"].astype(str).tolist())
    #ax.set_xlabel("LLI (median) ± bootstrap 95% CI | positive: SNAP leads")
    lead_txt = "MCP leads" if LLI_SIGN == -1 else "SNAP leads"
    ax.set_xlabel(f"LLI (median) ± bootstrap 95% CI | positive: {lead_txt}")

    ax.set_title("Selected factors: LLI + time-shift permutation (p/q)", fontsize=FIGCFG["title_fontsize"])

    # xlim with padding
    xmin = np.nanmin(lo)
    xmax = np.nanmax(hi)
    pad = (xmax - xmin) * float(FIGCFG.get("forest_xpad", 0.05))
    right_extra = 1 * (xmax - xmin)  # 注釈用の追加余白
    ax.set_xlim(xmin - pad, xmax + pad + right_extra)
    # if np.isfinite(xmin) and np.isfinite(xmax) and xmin != xmax:
    #     pad = (xmax - xmin) * float(FIGCFG.get("forest_xpad", 0.05))
    #     ax.set_xlim(xmin - pad, xmax + pad)

    # p/q text at right side
    for i, row in df.iterrows():
        txt = f"N={int(row['n_cells'])}, p={_fmt_float(row['p_perm_two'])}, q={_fmt_float(row['q_perm_two'])}"
        ax.text(
            0.98, i,
            txt,
            transform=ax.get_yaxis_transform(),  # x in axes, y in data
            ha="right", va="center",
        )

    ax.grid(axis="x", linestyle=":", alpha=0.3)

    fig.tight_layout()
    fig.savefig(pdf_path, bbox_inches="tight")
    if FIGCFG.get("show_inline", True):
        plt.show()
    plt.close(fig)
    print("Saved:", pdf_path.resolve())


def export_evidence_pdf(pdf_path: Path, factors):
    # split pages if too many rows
    per_page = int(FIGCFG.get("evidence_rows_per_page", 4))
    per_page = max(1, per_page)

    def chunks(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]

    with PdfPages(pdf_path) as pp:
        for chunk in chunks(list(factors), per_page):
            nrows = len(chunk)
            fig_h = FIGCFG["evidence_row_height"] * nrows
            fig_w = FIGCFG["evidence_fig_width"]

            fig = plt.figure(figsize=(fig_w, fig_h))
            gs = gridspec.GridSpec(
                nrows=nrows, ncols=2,
                width_ratios=[1.25, 1.0],
                hspace=FIGCFG.get("evidence_hspace", 0.55),
                wspace=FIGCFG.get("evidence_wspace", 0.25),
            )

            for r, fac in enumerate(chunk):
                ds = datasets.get(fac, None)
                if ds is None:
                    # compute on the fly (in case not precomputed)
                    row = get_row_for_factor(lli_summary_ok, fac)
                    ds = precompute_factor_dataset(fac, row, recompute=False, verbose=True)
                    datasets[fac] = ds

                axL = fig.add_subplot(gs[r, 0])
                axR = fig.add_subplot(gs[r, 1])

                lli_row = get_row_for_factor(lli_summary_ok, fac)
                N = int(lli_row["n_cells"])
                title = f"{fac} (N = {N}): mean cross-corr ±95% CI"
                plot_crosscorr_panel(axL, ds, annotate=False, title=title, lli_row=lli_row)

                plot_null_hist_panel(axR, ds, show_legend=(r == 0 and len(chunk) == 1))

            fig.suptitle(
                "Lead/Lag evidence: cross-corr + LLI time-shift (fast precompute version)",
                fontsize=FIGCFG["title_fontsize"],
                y=0.995,
            )
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pp.savefig(fig, bbox_inches="tight")
            if FIGCFG.get("show_inline", True):
                plt.show()
            plt.close(fig)

    print("Saved:", pdf_path.resolve())


## 7) 実行：PDF を生成する

ここまで設定した `FIGCFG` と factor リストに従って PDF を出力します。

- `final_figs/Fig_LLI_forest.pdf`
- `final_figs/Fig_LLI_crosscorr_evidence.pdf`

出力先は `FIGCFG["outdir"]` を変更すれば任意に指定できます。


In [ ]:
# ============================================================
# 7) Run export
# ============================================================

forest_pdf_path = FIG_OUTDIR / FIGCFG["forest_pdf"]
evidence_pdf_path = FIG_OUTDIR / FIGCFG["evidence_pdf"]

export_forest_pdf(forest_pdf_path, forest_factors)
export_evidence_pdf(evidence_pdf_path, evidence_factors)


In [ ]:
# === Compose figures: settings ===
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# 事前チェック（このセルは「最終図作成」セルの後に実行する想定）
_required = ["lli_summary_ok", "datasets", "get_row_for_factor", "precompute_factor_dataset"]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "このセルは、直前の『最終図（forest+evidence）作成』セルを実行した後に動かす想定である。\n"
        f"Missing: {_missing}"
    )

# LLI_SIGN が無い場合はデフォルト（あなたの現在の表示方針に合わせて -1）
# （既に定義済みなら、その値を優先）
if "LLI_SIGN" not in globals():
    LLI_SIGN = -1

# 出力先（既に FIG_OUTDIR があればそれを使い、無ければカレントに出す）
if "FIG_OUTDIR" in globals():
    _outdir_default = Path(FIG_OUTDIR)
else:
    _outdir_default = Path("final_figs_extra")
_outdir_default.mkdir(parents=True, exist_ok=True)

# ここを編集すればOK
COMPOSE = {
    # 出力
    "outdir": _outdir_default,
    "main_pdf": "Fig_main_panel_a_overlay_panel_b_forest.pdf",
    "supp_overlay_pdf": "FigS_overlay_BRD4_SIN3A.pdf",
    "supp_nullgrid_pdf": "FigS_null_hist_2x2.pdf",

    # 図サイズ（inch）
    "main_figsize": (9, 3),        # a+b 横並び
    "supp_overlay_figsize": (4.5, 3), # overlay単独
    "supp_nullgrid_figsize": (9.0, 6),

    # パネルa（Main）のoverlay因子
    "main_overlay_factors": ["SOX2", "HDAC3"],

    # パネルb（Main）のforestに載せる因子
    "main_forest_factors": ["SOX2", "BRD4", "SIN3A", "HDAC3"],

    # Supp.1 overlay因子
    "supp_overlay_factors": ["BRD4", "SIN3A"],

    # Supp.2 2x2 histogram 因子（順に左上→右上→左下→右下）
    "supp_nullgrid_factors": ["SOX2", "BRD4", "SIN3A", "HDAC3"],

    # 見た目
    "overlay_ci_alpha": 0.15,
    "overlay_linewidth": 2.2,

    # panel label
    "panel_label_fontsize": 18,
}
print("COMPOSE outdir =", Path(COMPOSE["outdir"]).resolve())


In [ ]:
# === Compose figures: helpers ===

def _ensure_dataset(factor: str):
    """datasets に無ければ precompute して補う"""
    factor = str(factor)
    if factor in datasets:
        return datasets[factor]
    row = get_row_for_factor(lli_summary_ok, factor)
    ds = precompute_factor_dataset(factor, row, recompute=False, verbose=True)
    datasets[factor] = ds
    return ds

def _draw_overlay_crosscorr(ax, factors, title=None):
    """
    mean cross-corr ±95%CI を同一軸に重ね描きする
    factors: list[str]（通常2つを想定）
    """
    factors = [str(x) for x in factors]
    if len(factors) < 1:
        raise ValueError("factors が空である。")

    # 先にLLI windowの薄灰（dtがあれば）を敷く
    ds0 = _ensure_dataset(factors[0])
    dt = ds0.get("dt_min", np.nan)
    win = int(ds0.get("win_frames", 5))
    if ("FIGCFG" in globals()) and FIGCFG.get("shade_lli_window", True) and np.isfinite(dt):
        dt = float(dt)
        ax.axvspan(-win * dt, -1 * dt, color="0.92", zorder=0)
        ax.axvspan( 1 * dt,  win * dt, color="0.92", zorder=0)

    # 重ね描き（Matplotlibのデフォルトカラ―サイクルを利用）
    for fac in factors:
        ds = _ensure_dataset(fac)
        t = ds["t"]
        m = ds["mean_curve"]
        lo = ds["ci_lo"]
        hi = ds["ci_hi"]

        line = ax.plot(t, m, lw=COMPOSE["overlay_linewidth"], label=fac)[0]
        c = line.get_color()
        ax.fill_between(t, lo, hi, alpha=COMPOSE["overlay_ci_alpha"], color=c)

    ax.axvline(0, color="black", lw=1.0, ls=":")
    ax.axhline(0, color="black", lw=1.0, ls="--")
    ax.grid(axis="y", linestyle=":", alpha=0.6)

    ax.set_xlabel(ds0.get("xlab", "Lag τ"))
    ax.set_ylabel("Cross-corr (baseline-corrected)")

    if title is None:
        title = " & ".join(factors) + ": mean cross-corr ±95% CI"
    ax.set_title(title)
    ax.legend(frameon=False, loc="best")

def _draw_forest_on_ax(ax, factors, title=None):
    """
    export_forest_pdf 相当を、既存figureのaxに描画する（合成用）
    """
    factors = [str(x) for x in factors]
    df = lli_summary_ok.set_index("factor").loc[factors].reset_index()

    n = len(df)
    y = np.arange(n, dtype=float)
    ax.invert_yaxis()

    x = LLI_SIGN * df["lli_median"].astype(float).to_numpy()
    lo0 = df["lli_ci_low"].astype(float).to_numpy()
    hi0 = df["lli_ci_high"].astype(float).to_numpy()

    # 符号反転後のCI端点（export_forest_pdfと同じ処理）
    lo = LLI_SIGN * hi0
    hi = LLI_SIGN * lo0

    xerr = np.vstack([x - lo, hi - x])
    ax.errorbar(x, y, xerr=xerr, fmt="o", color="black", ecolor="black", capsize=3)
    ax.axvline(0, color="black", lw=1.2, ls="--")
    ax.grid(axis="x", linestyle=":", alpha=0.3)

    ax.set_yticks(y)
    ax.set_yticklabels(df["factor"].astype(str).tolist())

    lead_txt = "MCP leads" if LLI_SIGN == -1 else "SNAP leads"
    ax.set_xlabel(f"LLI (median) ± bootstrap 95% CI | positive: {lead_txt}")

    if title is None:
        title = "Selected factors: LLI + time-shift permutation (p/q)"
    ax.set_title(title)

    # xlim（右側注釈の余白込み）
    xmin = np.nanmin(lo)
    xmax = np.nanmax(hi)
    rng = xmax - xmin
    if (not np.isfinite(rng)) or rng == 0:
        rng = 1.0
    pad = rng * 0.10
    right_extra = rng * 1.25
    ax.set_xlim(xmin - pad, xmax + pad + right_extra)

    # 右側注釈（N, p, q）
    for i, row in df.iterrows():
        txt = f"n={int(row['n_cells'])}, p={_fmt_float(row['p_perm_two'])}, q={_fmt_float(row['q_perm_two'])}"
        ax.text(
            0.98, i,
            txt,
            transform=ax.get_yaxis_transform(),  # xはaxes, yはdata
            ha="right", va="center",
        )

def _draw_null_hist_grid(fig, axes, factors):
    """
    export_evidence_pdf の「右側」を2x2等に並べる
    """
    factors = [str(x) for x in factors]
    for ax, fac in zip(np.ravel(axes), factors):
        ds = _ensure_dataset(fac)
        plot_null_hist_panel(ax, ds)

        # 追加でN/p/qを入れておく（任意）
        row = get_row_for_factor(lli_summary_ok, fac)
        txt = f"n={int(row['n_cells'])}\np={_fmt_float(row['p_perm_two'])}\nq={_fmt_float(row['q_perm_two'])}"
        ax.text(
            0.02, 0.98, txt,
            transform=ax.transAxes,
            ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.95),
        )

    # 余ったaxesがあれば消す
    flat = list(np.ravel(axes))
    if len(flat) > len(factors):
        for ax in flat[len(factors):]:
            ax.axis("off")


# === 1) Main figure (panel a + b) ===
outdir = Path(COMPOSE["outdir"])
outdir.mkdir(parents=True, exist_ok=True)

main_pdf = outdir / COMPOSE["main_pdf"]
fig = plt.figure(figsize=COMPOSE["main_figsize"], constrained_layout=True)
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.1, 1.30], wspace=0.1)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])

_draw_overlay_crosscorr(
    ax_a,
    COMPOSE["main_overlay_factors"],
    title="Mean cross-correlation ±95% CI"
)
_draw_forest_on_ax(
    ax_b,
    COMPOSE["main_forest_factors"],
    title="LLI + time-shift permutation"
)

# パネルラベル（a, b）
ax_a.text(
    -0.18, 1.15, "",
    transform=ax_a.transAxes,
    fontsize=COMPOSE["panel_label_fontsize"],
    fontweight="bold",
    va="top"
)
ax_b.text(
    -0.18, 1.15, "",
    transform=ax_b.transAxes,
    fontsize=COMPOSE["panel_label_fontsize"],
    fontweight="bold",
    va="top"
)

fig.savefig(main_pdf, bbox_inches="tight")
if ("FIGCFG" in globals()) and FIGCFG.get("show_inline", True):
    plt.show()
plt.close(fig)
print("Saved:", main_pdf.resolve())


# === 2) Supplementary overlay (BRD4 + SIN3A) ===
supp_overlay_pdf = outdir / COMPOSE["supp_overlay_pdf"]
fig, ax = plt.subplots(figsize=COMPOSE["supp_overlay_figsize"], constrained_layout=True)
_draw_overlay_crosscorr(
    ax,
    COMPOSE["supp_overlay_factors"],
    title="BRD4 and SIN3A (mean cross-corr ±95% CI)"
)
fig.savefig(supp_overlay_pdf, bbox_inches="tight")
if ("FIGCFG" in globals()) and FIGCFG.get("show_inline", True):
    plt.show()
plt.close(fig)
print("Saved:", supp_overlay_pdf.resolve())


# === 3) Supplementary null histograms in 2x2 ===
supp_nullgrid_pdf = outdir / COMPOSE["supp_nullgrid_pdf"]
facs = COMPOSE["supp_nullgrid_factors"]

fig, axes = plt.subplots(2, 2, figsize=COMPOSE["supp_nullgrid_figsize"], constrained_layout=True)
_draw_null_hist_grid(fig, axes, facs)

fig.suptitle("Time-shift null distributions (right panels of evidence)", y=1.02)
fig.savefig(supp_nullgrid_pdf, bbox_inches="tight")
if ("FIGCFG" in globals()) and FIGCFG.get("show_inline", True):
    plt.show()
plt.close(fig)
print("Saved:", supp_nullgrid_pdf.resolve())
